# Demand Forecasting

In this notebook, we forecast weekly SKU-level demand.
Forecasting is done in a segment-aware manner, based on the ABC/XYZ
classification created earlier.

The goal is not just low error, but forecasts that are stable and usable
for inventory planning.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.linear_model import LinearRegression

from src.forecasting import (
    train_test_split_time,
    naive_forecast,
    prepare_lag_features,
    evaluate_forecast
)

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("data/processed/feature_engineered_with_segments.csv")
df["date"] = pd.to_datetime(df["date"])

## Weekly Aggregation

Inventory replenishment is planned weekly, so we aggregate demand
to weekly SKU-level data.

In [ ]:
weekly_df = (
    df
    .set_index("date")
    .groupby(["sku_id", "SKU_segment"])
    .resample("W")["units_sold"]
    .sum()
    .reset_index()
)

weekly_df.head()

## Train-Test Split

We use a time-based split to avoid data leakage.
The last 20% of weeks are used for testing.

In [ ]:
def train_test_split_time(series, test_size=0.2):
    split_idx = int(len(series) * (1 - test_size))
    return series.iloc[:split_idx], series.iloc[split_idx:]

## Baseline Model: Naive Forecast

The naive model uses last week’s demand as the forecast.
Any advanced model must beat this baseline.

In [ ]:
def naive_forecast(train, test):
    return np.repeat(train.iloc[-1], len(test))

## Lag-Based Regression Model

A simple regression model using lagged demand as a predictor.
This works well for stable and moderately seasonal SKUs.

In [ ]:
def prepare_lag_features(series, lags=[1, 2, 4]):
    data = pd.DataFrame({"y": series})
    for lag in lags:
        data[f"lag_{lag}"] = series.shift(lag)
    return data.dropna()

## Evaluation Metrics

We evaluate models using:
- MAPE (relative error)
- RMSE (absolute error)

In [ ]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False)
    }

## Forecasting on Sample SKUs

For demonstration, we apply forecasting to a subset of SKUs
from different segments.

In [ ]:
results = []

sample_skus = weekly_df["sku_id"].unique()[:10]

for sku in sample_skus:
    sku_data = weekly_df[weekly_df["sku_id"] == sku].sort_values("date")
    series = sku_data["units_sold"].reset_index(drop=True)

    if len(series) < 20:
        continue

    train, test = train_test_split_time(series)

    # Naive
    naive_pred = naive_forecast(train, test)
    naive_metrics = evaluate_forecast(test, naive_pred)

    # Regression
    lagged = prepare_lag_features(series)
    train_lag, test_lag = train_test_split_time(lagged)

    X_train = train_lag.drop("y", axis=1)
    y_train = train_lag["y"]
    X_test = test_lag.drop("y", axis=1)
    y_test = test_lag["y"]

    model = LinearRegression()
    model.fit(X_train, y_train)
    reg_pred = model.predict(X_test)

    reg_metrics = evaluate_forecast(y_test, reg_pred)

    results.append({
        "sku_id": sku,
        "SKU_segment": sku_data["SKU_segment"].iloc[0],
        "Naive_MAPE": naive_metrics["MAPE"],
        "Reg_MAPE": reg_metrics["MAPE"],
        "Naive_RMSE": naive_metrics["RMSE"],
        "Reg_RMSE": reg_metrics["RMSE"]
    })

## Model Comparison Results

In [ ]:
results_df = pd.DataFrame(results)
results_df

## Key Forecasting Insights

- Complex models are not always better than simple baselines
- Demand volatility limits forecast accuracy more than model choice
- Forecast error must be explicitly accounted for in inventory decisions

These insights directly inform safety stock calculations
in the next notebook.

## Save Forecast Error Summary

Forecast error statistics will be used to compute safety stock.

In [ ]:
results_df.to_csv(
    "data/processed/forecast_error_summary.csv",
    index=False
)